In [2]:
%load_ext autoreload
%autoreload 2

# INITIALIZATION STEPS

# CHROMA DB TEST

In [3]:
import chromadb

print(chromadb.__version__)

client = chromadb.PersistentClient(path="./chroma_db")
test = client.get_or_create_collection("test")
test.add(ids=["1"], documents=["hello world"])
print(test.query(query_texts=["hello"], n_results = 1))

1.5.5
{'ids': [['1']], 'embeddings': None, 'documents': [['hello world']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[0.4976864159107208]]}


# CHROMA STORE TEST

In [ ]:
import sys
sys.path.append("../")  # lets the notebook find the rag/module

from rag.chroma import ChromaStore

store = ChromaStore()
print("Collections loaded:", list(store.collections.keys()))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10071.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collections loaded: ['quests', 'lore']


# INDEXER TEST

In [6]:
import sys
sys.path.append("../")

from rag.chroma import ChromaStore
from rag.indexer import Indexer

store = ChromaStore()
indexer = Indexer(store)

# add_raw test
indexer.add_raw(
    collection="quests",
    id="q_test_001",
    text="Eat Food: You are hungry. Find something to eat.",
    metadata={"source": "test"}
)
print("After add_raw:", store.count("quests"))

# add_bulk test
indexer.add_bulk("lore", [
    {"id": "l_test_001", "text": "Hokma: deez nuts", "metadata": {"source": "test"}},
    {"id": "l_test_002", "text": "Shmebulock: shmebulock!", "metadata": {"source": "test"}},
])
print("After add_bulk:", store.count("lore"))

# upsert (i.e., same id, updated text)
indexer.add_raw(
    collection="quests",
    id="q_test_001",
    text="Obtain Kromer: Consume kromer.",
    metadata={"source": "test"}
)
print("After upsert:", store.count("quests"))
print("Updated text:", store.query("quests", "Lost Sword")[0]["text"])

# cleanup 
store.delete("quests", ["q_test_001"])
store.delete("lore", ["l_test_001", "l_test_002"])
print("Quests after cleanup:", store.count("quests"))
print("Lore after cleanup:", store.count("lore"))

After add_raw: 1
After add_bulk: 2
After upsert: 1
Updated text: Obtain Kromer: Consume kromer.
Quests after cleanup: 0
Lore after cleanup: 0


# RETRIEVER TEST

In [15]:
import sys
sys.path.append("../")

from rag.chroma import ChromaStore
from rag.indexer import Indexer
from rag.retriever import Retriever

store = ChromaStore()
indexer = Indexer(store)
retriever = Retriever(store)

# test data 
indexer.add_bulk("quests", [
    {"id": "q_test_001", "text": "Eat Food: You are hungry. Find something to eat.", "metadata": {"source": "test"}},
    {"id": "q_test_002", "text": "Obtain Kromer: Consume kromer.", "metadata": {"source": "test"}},
])
indexer.add_bulk("lore", [
    {"id": "l_test_001", "text": "Hokma: deez nuts", "metadata": {"source": "test"}},
    {"id": "l_test_002", "text": "Shmebulock: shmebulock!", "metadata": {"source": "test"}},
])

# test query 
results = retriever.query("I am Hokma and I am in need of bread.")

for r in results:
    print(f"[{r['collection']}] Score: {r['score']:.2f} | {r['text']}")

# cleanup 
store.delete("quests", ["q_test_001", "q_test_002"])
store.delete("lore", ["l_test_001", "l_test_002"])
print("\nCleaned up.")

[lore] Score: 0.51 | Hokma: deez nuts
[quests] Score: 0.35 | Eat Food: You are hungry. Find something to eat.

Cleaned up.
